# 103 — Transformación y descomposición de consultas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

La consulta del usuario rara vez es una buena consulta de recuperación. Capa de
transformación entre usuario y retriever:

- **Rewriting** (arXiv:2305.14283): resolver correferencias con el historial, limpiar
  ruido, expandir siglas. 1 llamada LLM, casi siempre rentable en conversación.
- **Multi-query**: generar m paráfrasis, recuperar con cada una, fusionar con RRF
  (clase 100). Sube recall a cambio de m búsquedas.
- **HyDE** (arXiv:2212.10496): embeber una **respuesta hipotética** escrita por el LLM
  en vez de la pregunta. Funciona aunque el hipotético sea factualmente falso: se
  aprovecha su forma y vocabulario, que se parecen a los documentos reales.
- **Descomposición** (least-to-most, arXiv:2205.10625): partir una pregunta multi-hop en
  sub-preguntas secuenciales; la respuesta de una alimenta la siguiente. Riesgo:
  propagación de errores.
- **Step-back** (arXiv:2310.06117): recuperar también para la versión general de la
  pregunta (principios) además de la específica.
- **Routing**: clasificar cada consulta hacia la fuente/técnica adecuada; punto único
  de fallo que exige registro por consulta.

## 🧮 Ejemplo de referencia

```text
Q: "¿Qué edad tenía el fundador de la empresa que compró Instagram
    cuando lanzó su primer producto?"

1. ¿Qué empresa compró Instagram?        → Facebook (2012)
2. ¿Quién fundó Facebook?                → Mark Zuckerberg
3. ¿Cuándo lanzó su primer producto?     → 2004
4. ¿En qué año nació?                    → 1984
Síntesis: 2004 − 1984 = 20 años
```

Sin descomponer, el retriever busca un único pasaje con toda la cadena — que no suele
existir. Coste: 4 recuperaciones + 4 llamadas; si el paso 1 falla, todo lo posterior es
coherentemente erróneo.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("workflow", seed=103)
show(result)


## Reflexión

1. ¿Por qué HyDE puede mejorar la recuperación incluso cuando el documento hipotético contiene datos falsos, y en qué caso concreto empeora el resultado?
2. En una descomposición de 4 pasos donde cada paso acierta con probabilidad 0.9, ¿cuál es la probabilidad aproximada de que la cadena completa sea correcta, y qué mecanismo la mejoraría?
3. ¿Qué evidencia necesitarías para justificar añadir multi-query (3 variantes) a un pipeline que ya funciona: qué métrica debería subir y qué costes deberías reportar junto a ella?